In [1]:
from dotenv import load_dotenv
load_dotenv()

from ragaai_catalyst.tracers import Tracer
from ragaai_catalyst import RagaAICatalyst, init_tracing
from ragaai_catalyst import trace_tool, trace_llm, trace_agent

import os
from llama_index.core import (
    VectorStoreIndex,
    SimpleKeywordTableIndex,
    SimpleDirectoryReader,
    SummaryIndex,
)
from llama_index.core.schema import IndexNode
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.llms.openai import OpenAI
from llama_index.core.callbacks import CallbackManager
from llama_index.core.node_parser import SentenceSplitter
from llama_index.agent.openai import OpenAIAgent
from llama_index.core import load_index_from_storage, StorageContext
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

# Initialize RagaAI Catalyst
catalyst = RagaAICatalyst(
    access_key=os.getenv("RAGAAI_CATALYST_ACCESS_KEY"),
    secret_key=os.getenv("RAGAAI_CATALYST_SECRET_KEY"),
    base_url=os.getenv("RAGAAI_CATALYST_BASE_URL"),
)
# project = catalyst.create_project(
#     project_name="multi_doc_agent5",
#     usecase="Agentic Application"
# )

# Get project usecases
catalyst.project_use_cases()

# List projects
projects = catalyst.list_projects()
print(projects)
# Initialize tracer with metadata
tracer = Tracer(
    project_name="multi_doc_agent5",
    dataset_name="wiki_docs",
    tracer_type="Agentic",
    metadata={
        "description": "Multi-document agent for city information retrieval",
        "version": "1.0",
        "environment": "development"
    }
)

init_tracing(catalyst=catalyst, tracer=tracer)

@trace_tool("wiki_data_loader")
def load_wiki_data(wiki_titles):
    """Load Wikipedia data for given titles"""
    try:
        import requests
        from pathlib import Path
        
        loaded_data = []
        for title in wiki_titles:
            response = requests.get(
                "https://en.wikipedia.org/w/api.php",
                params={
                    "action": "query",
                    "format": "json",
                    "titles": title,
                    "prop": "extracts",
                    "explaintext": True,
                },
            ).json()
            page = next(iter(response["query"]["pages"].values()))
            wiki_text = page["extract"]

            data_path = Path("data")
            if not data_path.exists():
                Path.mkdir(data_path)

            with open(data_path / f"{title}.txt", "w") as fp:
                fp.write(wiki_text)
            
            loaded_data.append({
                "title": title,
                "text_length": len(wiki_text),
                "file_path": str(data_path / f"{title}.txt")
            })
        
        return {
            "tool_type": "wiki_loader",
            "cities_loaded": len(wiki_titles),
            "data_details": loaded_data
        }
    except Exception as e:
        print(f"Error loading wiki data: {str(e)}")
        raise

@trace_tool("document_loader")
def load_city_documents(wiki_titles):
    """Load documents for each city"""
    try:
        city_docs = {}
        doc_info = []
        for wiki_title in wiki_titles:
            docs = SimpleDirectoryReader(
                input_files=[f"data/{wiki_title}.txt"]
            ).load_data()
            city_docs[wiki_title] = docs
            doc_info.append({
                "city": wiki_title,
                "doc_count": len(docs),
                "file_path": f"data/{wiki_title}.txt"
            })
        
        return {
            "tool_type": "document_loader",
            "total_cities": len(wiki_titles),
            "documents_loaded": doc_info,
            "city_docs": city_docs
        }
    except Exception as e:
        print(f"Error loading city documents: {str(e)}")
        raise

@trace_tool("index_builder")
def build_indices(wiki_titles, city_docs):
    """Build vector and summary indices for each city"""
    try:
        node_parser = SentenceSplitter()
        agents = {}
        query_engines = {}
        all_nodes = []
        index_info = []

        for wiki_title in wiki_titles:
            nodes = node_parser.get_nodes_from_documents(city_docs[wiki_title])
            all_nodes.extend(nodes)

            if not os.path.exists(f"./data/{wiki_title}"):
                vector_index = VectorStoreIndex(nodes)
                vector_index.storage_context.persist(persist_dir=f"./data/{wiki_title}")
            else:
                vector_index = load_index_from_storage(
                    StorageContext.from_defaults(persist_dir=f"./data/{wiki_title}"),
                )

            summary_index = SummaryIndex(nodes)
            vector_query_engine = vector_index.as_query_engine(llm=Settings.llm)
            summary_query_engine = summary_index.as_query_engine(llm=Settings.llm)

            tools_info = create_query_tools(wiki_title, vector_query_engine, summary_query_engine)
            agent_info = create_agent(tools_info["tools"], wiki_title)
            
            agents[wiki_title] = agent_info["agent"]
            query_engines[wiki_title] = vector_index.as_query_engine(similarity_top_k=2)
            
            index_info.append({
                "city": wiki_title,
                "node_count": len(nodes),
                "has_vector_index": True,
                "has_summary_index": True
            })

        return {
            "tool_type": "index_builder",
            "total_nodes": len(all_nodes),
            "indices_created": index_info,
            "result": (agents, query_engines, all_nodes)
        }
    except Exception as e:
        print(f"Error building indices: {str(e)}")
        raise

@trace_tool("query_tools_creator")
def create_query_tools(wiki_title, vector_query_engine, summary_query_engine):
    """Create query engine tools for a city"""
    try:
        tools = [
            QueryEngineTool(
                query_engine=vector_query_engine,
                metadata=ToolMetadata(
                    name="vector_tool",
                    description=f"Useful for questions related to specific aspects of {wiki_title}",
                ),
            ),
            QueryEngineTool(
                query_engine=summary_query_engine,
                metadata=ToolMetadata(
                    name="summary_tool",
                    description=f"Useful for holistic summaries about {wiki_title}",
                ),
            ),
        ]
        
        return {
            "tool_type": "query_tools",
            "city": wiki_title,
            "tools_created": [
                {"name": "vector_tool", "type": "vector_query"},
                {"name": "summary_tool", "type": "summary_query"}
            ],
            "tools": tools
        }
    except Exception as e:
        print(f"Error creating query tools for {wiki_title}: {str(e)}")
        raise

@trace_agent("city_agent")
def create_agent(query_engine_tools, wiki_title):
    """Create an agent for a specific city"""
    try:
        function_llm = OpenAI(model="gpt-4")
        agent = OpenAIAgent.from_tools(
            query_engine_tools,
            llm=function_llm,
            verbose=True,
            system_prompt=f"You are a specialized agent designed to answer queries about {wiki_title}.",
        )
        
        return {
            "agent_type": "city_agent",
            "city": wiki_title,
            "tools_count": len(query_engine_tools),
            "llm_model": "gpt-4",
            "system_prompt": f"You are a specialized agent designed to answer queries about {wiki_title}.",
            "agent": agent
        }
    except Exception as e:
        print(f"Error creating agent for {wiki_title}: {str(e)}")
        raise

@trace_tool("top_agent_tools")
def create_top_agent_tools(agents, wiki_titles):
    """Create tools for the top-level agent"""
    try:
        all_tools = []
        tool_info = []
        
        for wiki_title in wiki_titles:
            wiki_summary = f"This tool provides information about {wiki_title}."
            doc_tool = QueryEngineTool(
                query_engine=agents[wiki_title],
                metadata=ToolMetadata(
                    name=f"tool_{wiki_title}",
                    description=wiki_summary,
                ),
            )
            all_tools.append(doc_tool)
            tool_info.append({
                "city": wiki_title,
                "tool_name": f"tool_{wiki_title}",
                "description": wiki_summary
            })
        
        return {
            "tool_type": "top_agent_tools",
            "total_tools": len(all_tools),
            "tools_created": tool_info,
            "tools": all_tools
        }
    except Exception as e:
        print(f"Error creating top agent tools: {str(e)}")
        raise

@trace_llm("query_processor")
def process_query(agent, query):
    """Process a query using the agent"""
    try:
        response = agent.query(query)
        return {
            "query": query,
            "response": str(response),
            "success": True
        }
    except Exception as e:
        print(f"Error processing query: {str(e)}")
        return {
            "query": query,
            "error": str(e),
            "success": False
        }

def main():
    wiki_titles = [
        "Toronto", "Seattle", "Chicago", "Boston", "Houston",
        "Tokyo", "Berlin", "Lisbon", "Paris", "London",
        "Atlanta", "Munich", "Shanghai", "Beijing",
        "Copenhagen", "Moscow", "Cairo", "Karachi",
    ]

    with tracer:
        try:
            # Load wiki data
            wiki_data_result = load_wiki_data(wiki_titles)
            
            # Load documents
            docs_result = load_city_documents(wiki_titles)
            city_docs = docs_result["city_docs"]
            
            # Initialize LLM settings
            Settings.llm = OpenAI(temperature=0, model="gpt-3.5-turbo")
            Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")
            
            # Build indices and agents
            indices_result = build_indices(wiki_titles, city_docs)
            agents, query_engines, all_nodes = indices_result["result"]
            
            # Create top agent tools
            tools_result = create_top_agent_tools(agents, wiki_titles)
            all_tools = tools_result["tools"]
            
            # Create object index
            obj_index = ObjectIndex.from_objects(
                all_tools,
                index_cls=VectorStoreIndex,
            )
            
            # Create top agent
            top_agent = OpenAIAgent.from_tools(
                tool_retriever=obj_index.as_retriever(similarity_top_k=3),
                system_prompt="You are an agent designed to answer queries about cities.",
                verbose=True,
            )
            
            # Example queries
            queries = [
                "Tell me about the arts and culture in Boston",
                "Give me a summary of all the positive aspects of Houston",
                "Compare the demographics of Houston and Chicago",
                "Tell me the differences between Shanghai and Beijing in terms of history and economy"
            ]
            
            # Process queries
            for query in queries:
                query_result = process_query(top_agent, query)
                if query_result["success"]:
                    print(f"\nQuery: {query}")
                    print(f"Response: {query_result['response']}\n")
                    print("-" * 80)
                else:
                    print(f"\nError processing query '{query}': {query_result['error']}")

        except Exception as e:
            print(f"Error in execution: {str(e)}")
        finally:
            tracer.stop()

if __name__ == "__main__":
    main()

INFO:httpx:HTTP Request: GET https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json "HTTP/1.1 200 OK"


Token(s) set successfully
['LlamaIndex_Agent_Demo', 'multi_doc_agent5', 'multi_doc_agent4', 'test_1', 'latency-test-2', 'multi_doc_agent3', 'asdas', 'alteryx_testing', 'langgraph_testing_release', 'testing_platform', 'multi_doc_agent2', 'multi_doc_agent', 'zip_test', 'langchain_retriever_tracing_test', 'lang_agent-tree', 'llama_index_example4', 'llama_index_example3', 'llama_index_example2', 'llama_index_examples', 'llama_index_example', 'fixes', 'sad', 'ritika_json_test_agentic_v2', 'cost_latency_schema_sk', 'cost_latency_tokecn_sk', 'llamaindex_fix', 'json_upload_test', 'Alteryx-Copilot', 'alteryx_testing_t', 'DeepHealth-sample', 'DeepHealth-demo-sample', 'DeepHealth-demo', 'asd', 'deephealth', 'Deephealth_custom_metrics', 'append_metric_test_sk', 'rag2', 'agentic_test_wed', 'dataset-upload-deephealth1', 'dataset-upload-deephealth', 'llamaindex', 'ds-pytest-qa', 'debug_ragagent', 'Execute_Metric_Test1', 'Execute_Metric_Test', 'LLAMAINDEX-RIYA', 'Langgraph-Test', 'Langgraph-Riya', 'Gu

INFO:llama_index.core.indices.loading:Loading all indices.
INFO:llama_index.core.indices.loading:Loading all indices.
INFO:llama_index.core.indices.loading:Loading all indices.
INFO:llama_index.core.indices.loading:Loading all indices.
INFO:llama_index.core.indices.loading:Loading all indices.
INFO:llama_index.core.indices.loading:Loading all indices.
INFO:llama_index.core.indices.loading:Loading all indices.
INFO:llama_index.core.indices.loading:Loading all indices.
INFO:llama_index.core.indices.loading:Loading all indices.
INFO:llama_index.core.indices.loading:Loading all indices.
INFO:llama_index.core.indices.loading:Loading all indices.
INFO:llama_index.core.indices.loading:Loading all indices.
INFO:llama_index.core.indices.loading:Loading all indices.
INFO:llama_index.core.indices.loading:Loading all indices.
INFO:llama_index.core.indices.loading:Loading all indices.
INFO:llama_index.core.indices.loading:Loading all indices.
INFO:llama_index.core.indices.loading:Loading all indice

Error in execution: name 'ObjectIndex' is not defined


INFO:ragaai_catalyst.tracers.agentic_tracing.tracers.base: Traces saved successfully.
ERROR:ragaai_catalyst.dataset:Failed to retrieve projects list: HTTPSConnectionPool(host='llm-dev5.ragaai.ai', port=443): Read timed out. (read timeout=30)


Error fetching traces metrics: HTTPSConnectionPool(host='llm-dev5.ragaai.ai', port=443): Read timed out. (read timeout=30)
Uploading agentic traces...
Uploading code...
Dataset trace code inserted successfully
